# Multilayer Neural Networks: Experimental Analysis
This notebook provides a **step-by-step tutorial** on how depth, width, and activation functions affect MLP performance.

Each step includes clear explanations to support learning and reproducibility.

**Accessibility features:**
- Colour-blind friendly plots (`cividis`)
- Clear labels and readable fonts


## Step 1: Import Libraries
In this step, we import all required libraries.

- `numpy`: numerical operations
- `matplotlib`: visualisation
- `sklearn`: dataset generation, preprocessing, and modelling

These libraries provide the tools needed to build and evaluate neural networks efficiently.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

plt.rcParams.update({'font.size': 12})

## Step 2: Dataset Preparation
We generate a **non-linear dataset** using `make_moons`.

Why this dataset?
- It is not linearly separable
- It highlights the strength of neural networks
- It allows visualisation of decision boundaries

We then:
- Split into training (80%) and testing (20%) sets
- Standardise features for better training stability

In [ ]:
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Step 3: Training Function
This function defines and trains the MLP model.

Inputs:
- `hidden_layers`: controls depth and width
- `activation`: defines the activation function

Outputs:
- Trained model
- Training accuracy
- Testing accuracy

This modular design improves readability and reusability.

In [ ]:
def train_mlp(hidden_layers, activation):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        max_iter=1000,
        random_state=42
    )
    model.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))

    return model, train_acc, test_acc

## Step 4: Decision Boundary Visualisation
This function visualises how the model separates classes.

Why important?
- Helps understand model behaviour
- Shows how complexity changes with architecture

Accessibility features:
- Colour-blind friendly colormap (`cividis`)
- Clear axis labels and titles

In [ ]:
def plot_decision_boundary(model, title):
    x_min, x_max = X_test[:, 0].min() - 1, X_test[:, 0].max() + 1
    y_min, y_max = X_test[:, 1].min() - 1, X_test[:, 1].max() + 1

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )

    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.figure()
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='cividis')
    plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='cividis')

    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.title(title)

    plt.show()

## Step 5: Depth Experiment
We analyse how increasing the number of hidden layers affects performance.

Expectation:
- Shallow networks underfit
- Moderate depth improves performance
- Too much depth gives diminishing returns

In [ ]:
depth_configs = [(32,), (32, 32), (32, 32, 32)]

for config in depth_configs:
    model, train_acc, test_acc = train_mlp(config, 'relu')
    print(f"Depth {len(config)} | Train: {train_acc:.3f} | Test: {test_acc:.3f}")
    plot_decision_boundary(model, f"Depth = {len(config)}")

## Step 6: Width Experiment
We analyse how increasing neurons affects performance.

Expectation:
- Small width → underfitting
- Moderate width → best performance
- Large width → diminishing returns

In [ ]:
width_configs = [(16,), (64,), (128,)]

for config in width_configs:
    model, train_acc, test_acc = train_mlp(config, 'relu')
    print(f"Width {config[0]} | Train: {train_acc:.3f} | Test: {test_acc:.3f}")
    plot_decision_boundary(model, f"Width = {config[0]}")

## Step 7: Activation Function Experiment
We compare activation functions.

Expectation:
- ReLU performs best
- Tanh performs moderately
- Sigmoid performs worst due to vanishing gradients

In [ ]:
activations = ['relu', 'tanh', 'logistic']

for act in activations:
    model, train_acc, test_acc = train_mlp((64, 64), act)
    print(f"{act.upper()} | Train: {train_acc:.3f} | Test: {test_acc:.3f}")
    plot_decision_boundary(model, f"Activation = {act.upper()}")

## Step 8: Conclusion
- Depth improves performance up to a point
- Width increases capacity but saturates
- ReLU provides the most stable training

This experiment highlights the importance of balancing model complexity with dataset requirements.